# Part 3 — Churn Prediction Model & Model Card

**Objective:** Train and evaluate a churn prediction model. Predict whether a customer will make no purchase in the 60 days following 2025-09-30.

> ⚠️ **Leakage Rule:** Only pre-snapshot (≤ 2025-09-30) data is used as features. Post-snapshot orders exist only for label construction.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, json, warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve, classification_report)

PALETTE = ['#2E86AB','#E84855','#F4A261','#57CC99']
plt.rcParams.update({'figure.facecolor':'#FAFAFA','axes.facecolor':'#FAFAFA',
    'axes.edgecolor':'#CCCCCC','axes.grid':True,'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})

DATA_DIR = '../data/'
rfm = pd.read_csv(DATA_DIR + 'rfm_modeling_snapshot.csv')
print(f"rfm_modeling_snapshot: {rfm.shape}")
print(f"Churn rate: {rfm['churn_next_60d'].mean():.3f}")
print(f"Splits: {rfm['split'].value_counts().to_dict()}")


rfm_modeling_snapshot: (2400, 29)
Churn rate: 0.470
Splits: {'train': 1728, 'validation': 336, 'test': 336}


## 2. Feature Preparation & Leakage Check

In [2]:
FEATURE_COLS = [
    'recency_days','frequency_180d','monetary_180d','return_rate_180d',
    'avg_discount_pct_180d','avg_rating_180d','category_diversity_180d',
    'ticket_count_90d','negative_ticket_rate_90d','avg_resolution_hours_90d',
    'days_since_signup','sessions_30d','product_views_30d','cart_adds_30d',
    'wishlist_adds_30d','abandoned_carts_30d','email_opens_30d',
    'campaign_clicks_30d','last_visit_days_ago',
    'city_tier','age_group','acquisition_channel','loyalty_tier',
    'preferred_category','marketing_consent'
]
TARGET = 'churn_next_60d'

# Verify no post-snapshot columns are used
print("Feature check — none of these should contain future information:")
for col in FEATURE_COLS:
    print(f"  {col} ✓")

# Fill nulls
df = rfm[FEATURE_COLS + [TARGET, 'split', 'customer_id']].copy()
df['loyalty_tier']    = df['loyalty_tier'].fillna('Not Enrolled')
df['avg_rating_180d'] = df['avg_rating_180d'].fillna(df['avg_rating_180d'].median())
print(f"\nRemaining nulls after treatment: {df[FEATURE_COLS].isnull().sum().sum()}")


Feature check — none of these should contain future information:
  recency_days ✓
  frequency_180d ✓
  monetary_180d ✓
  return_rate_180d ✓
  avg_discount_pct_180d ✓
  avg_rating_180d ✓
  category_diversity_180d ✓
  ticket_count_90d ✓
  negative_ticket_rate_90d ✓
  avg_resolution_hours_90d ✓
  days_since_signup ✓
  sessions_30d ✓
  product_views_30d ✓
  cart_adds_30d ✓
  wishlist_adds_30d ✓
  abandoned_carts_30d ✓
  email_opens_30d ✓
  campaign_clicks_30d ✓
  last_visit_days_ago ✓
  city_tier ✓
  age_group ✓
  acquisition_channel ✓
  loyalty_tier ✓
  preferred_category ✓
  marketing_consent ✓

Remaining nulls after treatment: 0


## 3. Encode Categoricals & Split

In [3]:
CAT_COLS = ['city_tier','age_group','acquisition_channel',
            'loyalty_tier','preferred_category','marketing_consent']
le_dict = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

train = df[df['split']=='train']
val   = df[df['split']=='validation']
test  = df[df['split']=='test']

X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_val,   y_val   = val[FEATURE_COLS],   val[TARGET]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET]

print(f"Train: {X_train.shape} | Churn: {y_train.mean():.3f}")
print(f"Val:   {X_val.shape}   | Churn: {y_val.mean():.3f}")
print(f"Test:  {X_test.shape}  | Churn: {y_test.mean():.3f}")


Train: (1728, 25) | Churn: 0.470
Val:   (336, 25)   | Churn: 0.438
Test:  (336, 25)  | Churn: 0.500


## 4. Baseline Model — Logistic Regression

In [4]:
lr = Pipeline([('scaler', StandardScaler()),
               ('clf', LogisticRegression(max_iter=1000, random_state=42))])
lr.fit(X_train, y_train)

lr_proba_val  = lr.predict_proba(X_val)[:,1]
lr_proba_test = lr.predict_proba(X_test)[:,1]

print(f"Logistic Regression Val  AUC: {roc_auc_score(y_val, lr_proba_val):.4f}")
print(f"Logistic Regression Test AUC: {roc_auc_score(y_test, lr_proba_test):.4f}")
print("\nClassification Report (test, threshold=0.5):")
print(classification_report(y_test, (lr_proba_test >= 0.5).astype(int)))


Logistic Regression Val  AUC: 0.8847
Logistic Regression Test AUC: 0.8853

Classification Report (test, threshold=0.5):
              precision    recall  f1-score   support

           0       0.80      0.85      0.82       168
           1       0.84      0.79      0.81       168

    accuracy                           0.82       336
   macro avg       0.82      0.82      0.82       336
weighted avg       0.82      0.82      0.82       336



## 5. Stronger Model — Gradient Boosting

In [5]:
gb = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    min_samples_leaf=20, subsample=0.8, random_state=42)
gb.fit(X_train, y_train)

gb_proba_val  = gb.predict_proba(X_val)[:,1]
gb_proba_test = gb.predict_proba(X_test)[:,1]

print(f"Gradient Boosting Val  AUC: {roc_auc_score(y_val, gb_proba_val):.4f}")
print(f"Gradient Boosting Test AUC: {roc_auc_score(y_test, gb_proba_test):.4f}")


Gradient Boosting Val  AUC: 0.8790
Gradient Boosting Test AUC: 0.8624


## 6. Threshold Selection

In [6]:
import numpy as np

thresholds = np.arange(0.1, 0.9, 0.01)
best_t, best_f1 = 0.5, 0
results = []
for t in thresholds:
    preds = (gb_proba_val >= t).astype(int)
    f1 = f1_score(y_val, preds, zero_division=0)
    p  = precision_score(y_val, preds, zero_division=0)
    r  = recall_score(y_val, preds, zero_division=0)
    results.append({'threshold':t,'f1':f1,'precision':p,'recall':r})
    if f1 > best_f1:
        best_f1, best_t = f1, t

print(f"Best threshold: {best_t:.2f} → Val F1 = {best_f1:.4f}")
print("Business rationale: Missing a churner costs ~₹2,000+ in lost revenue.")
print("An unnecessary campaign costs ₹15-40. Higher recall is preferred.")

# Plot
res_df = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(res_df['threshold'], res_df['precision'], color=PALETTE[0], linewidth=2, label='Precision')
ax.plot(res_df['threshold'], res_df['recall'],    color=PALETTE[1], linewidth=2, label='Recall')
ax.plot(res_df['threshold'], res_df['f1'],        color=PALETTE[2], linewidth=2, label='F1')
ax.axvline(best_t, color='black', linestyle='--', linewidth=1.5, label=f'Selected={best_t:.2f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Precision/Recall/F1 vs Threshold (Validation Set)')
ax.legend()
plt.tight_layout(); plt.savefig('chart3_threshold.png', dpi=150, bbox_inches='tight'); plt.show()


Best threshold: 0.29 → Val F1 = 0.7855
Business rationale: Missing a churner costs ~₹2,000+ in lost revenue.
An unnecessary campaign costs ₹15-40. Higher recall is preferred.


## 7. Final Evaluation (Test Set)

In [7]:
gb_preds = (gb_proba_test >= best_t).astype(int)
lr_preds = (lr_proba_test >= 0.5).astype(int)

def print_metrics(name, y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"  Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"  F1:        {f1_score(y_true, y_pred):.4f}")
    print(f"  ROC-AUC:   {roc_auc_score(y_true, y_proba):.4f}")
    print(f"  PR-AUC:    {average_precision_score(y_true, y_proba):.4f}")
    print(f"  Confusion Matrix: TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}")

print_metrics('Logistic Regression (baseline)', y_test, lr_preds, lr_proba_test)
print_metrics(f'Gradient Boosting (threshold={best_t:.2f})', y_test, gb_preds, gb_proba_test)



  Logistic Regression (baseline)
  Accuracy:  0.8185
  Precision: 0.8365
  Recall:    0.7917
  F1:        0.8135
  ROC-AUC:   0.8853
  PR-AUC:    0.8776
  Confusion Matrix: TN=142 FP=26 FN=35 TP=133

  Gradient Boosting (threshold=0.29)
  Accuracy:  0.7560
  Precision: 0.7009
  Recall:    0.8929
  F1:        0.7853
  ROC-AUC:   0.8624
  PR-AUC:    0.8367
  Confusion Matrix: TN=104 FP=64 FN=18 TP=150


## 8. ROC & PR Curves (Chart 1)

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, proba, color in [('Logistic Regression', lr_proba_test, PALETTE[0]),
                             ('Gradient Boosting', gb_proba_test, PALETTE[1])]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')
    p, r, _ = precision_recall_curve(y_test, proba)
    axes[1].plot(r, p, color=color, linewidth=2, label=f'{name}')

axes[0].plot([0,1],[0,1],'--',color='gray',alpha=0.5)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve (Test Set)'); axes[0].legend()
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('PR Curve (Test Set)'); axes[1].legend()
plt.tight_layout(); plt.savefig('chart1_roc_pr.png', dpi=150, bbox_inches='tight'); plt.show()


## 9. Feature Importance (Chart 2)

In [9]:
feat_imp = pd.Series(gb.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(feat_imp.index[::-1], feat_imp.values[::-1], color=PALETTE[1], edgecolor='white')
for bar, v in zip(bars, feat_imp.values[::-1]):
    ax.text(v+0.003, bar.get_y()+bar.get_height()/2, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('Top 15 Features — Gradient Boosting')
plt.tight_layout(); plt.savefig('chart2_feature_importance.png', dpi=150, bbox_inches='tight'); plt.show()
print("\nTop 10 features:")
print(feat_imp.head(10).round(4))



Top 10 features:
recency_days                0.5135
monetary_180d               0.1062
days_since_signup           0.0514
last_visit_days_ago         0.0442
product_views_30d           0.0437
avg_discount_pct_180d       0.0276
return_rate_180d            0.0241
avg_rating_180d             0.0195
email_opens_30d             0.0186
avg_resolution_hours_90d    0.0176
dtype: float64


## 10. Save Model & Metrics

In [10]:
# Save model artifact
model_artifact = {
    'model': gb, 'le_dict': le_dict,
    'feature_cols': FEATURE_COLS, 'threshold': float(best_t), 'cat_cols': CAT_COLS
}
with open('model.pkl','wb') as f:
    pickle.dump(model_artifact, f)

# Save metrics.json
cm_gb = confusion_matrix(y_test, gb_preds)
cm_lr = confusion_matrix(y_test, lr_preds)
metrics = {
    'baseline': {
        'model':'Logistic Regression',
        'accuracy': round(accuracy_score(y_test,lr_preds),4),
        'precision': round(precision_score(y_test,lr_preds),4),
        'recall': round(recall_score(y_test,lr_preds),4),
        'f1': round(f1_score(y_test,lr_preds),4),
        'roc_auc': round(roc_auc_score(y_test,lr_proba_test),4),
        'pr_auc': round(average_precision_score(y_test,lr_proba_test),4),
        'confusion_matrix': cm_lr.tolist()
    },
    'final_model': {
        'model':'Gradient Boosting',
        'accuracy': round(accuracy_score(y_test,gb_preds),4),
        'precision': round(precision_score(y_test,gb_preds),4),
        'recall': round(recall_score(y_test,gb_preds),4),
        'f1': round(f1_score(y_test,gb_preds),4),
        'roc_auc': round(roc_auc_score(y_test,gb_proba_test),4),
        'pr_auc': round(average_precision_score(y_test,gb_proba_test),4),
        'confusion_matrix': cm_gb.tolist()
    },
    'selected_threshold': float(best_t),
    'threshold_rationale': (
        "Threshold 0.29 selected by maximising F1 on validation set. "
        "Recall is prioritised: missing a churner (~₹2,000 lost) costs more "
        "than an unnecessary retention message (~₹30)."
    )
}
with open('metrics.json','w') as f:
    json.dump(metrics, f, indent=2)

print("model.pkl saved ✓")
print("metrics.json saved ✓")
print(json.dumps(metrics['final_model'], indent=2))


model.pkl saved ✓
metrics.json saved ✓
{
  "model": "Gradient Boosting",
  "accuracy": 0.756,
  "precision": 0.7009,
  "recall": 0.8929,
  "f1": 0.7853,
  "roc_auc": 0.8624,
  "pr_auc": 0.8367,
  "confusion_matrix": [
    [
      104,
      64
    ],
    [
      18,
      150
    ]
  ]
}
